In [1]:
# Install the required packages for Vespa Python client and Docker
# !pip install pyvespa docker

In [2]:
import random
import pickle
from vespa.deployment import VespaCloud
import json
import unicodedata
from dataclasses import dataclass
from typing import Callable, Optional, Iterable, Dict
from vespa.application import Vespa
from time import time
from tqdm.auto import tqdm
import nest_asyncio
from vespa.evaluation import VespaEvaluator
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function,
    OnnxModel,
)

In [3]:
import requests
from pathlib import Path

# url = "https://huggingface.co/mixedbread-ai/mxbai-rerank-xsmall-v1/resolve/main/onnx/model_quantized.onnx"
local_model_path = "model/model_quantized.onnx"

# r = requests.get(url)
# Create path if it doesn't exist
# Path(local_model_path).parent.mkdir(parents=True, exist_ok=True)
# with open(local_model_path, "wb") as f:
#     f.write(r.content)
#     print(f"Downloaded model to {local_model_path}")

In [14]:
# nome da aplicação
application = "reranking"
tenant_name = "projetos-segundo"

In [5]:

schema = Schema(
    name="doc",
    mode="index",
    document=Document(
        fields=[
            Field(name="id", type="string", indexing=["summary", "attribute"]),
            Field(
                name="text",
                type="string",
                indexing=["index", "summary"],
                index="enable-bm25",
                bolding=True,
            ),
            # Embedding field for semantic search
            Field(
                name="embedding",
                type="tensor<float>(x[384])",
                indexing=[
                    'input text',
                    "embed e5",  # Specify the embedder component
                    "index",
                    "attribute",
                ],
                ann=HNSW(distance_metric="angular"),
                is_document_field=False,
            ),
            # Tokenized field for cross-encoder
            Field(
                name="text_tokens",
                type="tensor<float>(d0[512])",
                indexing=["input text", "embed tokenizer", "attribute", "summary"],
                is_document_field=False,
            ),
        ],
    ),
    fieldsets=[FieldSet(name="default", fields=["text"])],
    models=[
        OnnxModel(
            model_name="crossencoder",
            model_file_path=f"{local_model_path}",
            inputs={
                "input_ids": "input_ids",
                "attention_mask": "attention_mask",
            },
            outputs={"logits": "logits"},
        )
    ],
    rank_profiles=[
        RankProfile(name="bm25", first_phase="bm25(text)"),
        RankProfile(
            name="semantic",
            inputs=[("query(q)", "tensor<float>(x[384])")],
            first_phase="closeness(field, embedding)",
        ),
        RankProfile(
            name="fusion",
            inputs=[("query(q)", "tensor<float>(x[384])")],
            functions=[
                Function(name="bm25sum", expression="bm25(text)")
            ],
            first_phase="closeness(field, embedding)",
            global_phase=GlobalPhaseRanking(
                expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding))",
                rerank_count=1000,
            ),
        ),
        RankProfile(
            name="reranking",
            inputs=[
                ("query(q)", "tensor<float>(x[384])"),  # For semantic search
                ("query(q_tokens)", "tensor<float>(d0[64])")  # For cross-encoder
            ],
            functions=[
                Function(name="bm25sum", expression="bm25(text)"),
                Function(
                    name="input_ids",
                    expression="customTokenInputIds(1, 2, 512, query(q_tokens), attribute(text_tokens))",
                ),
                Function(
                    name="attention_mask",
                    expression="tokenAttentionMask(512, query(q_tokens), attribute(text_tokens))",
                ),
            ],
            # Use fusion-like ranking as first phase
            first_phase="closeness(field, embedding)",
            global_phase=GlobalPhaseRanking(
                rerank_count=10,
                # First do reciprocal rank fusion, then apply cross-encoder
                expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding)) + sigmoid(onnx(crossencoder).logits{d0:0,d1:0})",
            ),
            summary_features=[
                "query(q)",
                "query(q_tokens)",
                "input_ids",
                "attention_mask",
                "onnx(crossencoder).logits",
                "bm25sum",
                "closeness(field, embedding)",
            ],
        ),
    ],
)

In [10]:
components = [
        Component(
            # See https://docs.vespa.ai/en/reference/embedding-reference.html#huggingface-tokenizer-embedder
            id="tokenizer",
            type="hugging-face-tokenizer",
            parameters=[
                Parameter(
                    "model",
                    {
                        "url": "https://huggingface.co/mixedbread-ai/mxbai-rerank-xsmall-v1/raw/main/tokenizer.json"
                    },
                ),
            ],
        ),
        Component(
            id="e5",
            type="hugging-face-embedder",
            parameters=[
                Parameter(
                    "transformer-model",
                    {
                        "url": "https://github.com/vespa-engine/sample-apps/raw/master/examples/model-exporting/model/e5-small-v2-int8.onnx"
                    },
                ),
                Parameter(
                    "tokenizer-model",
                    {
                        "url": "https://raw.githubusercontent.com/vespa-engine/sample-apps/master/examples/model-exporting/model/tokenizer.json"
                    },
                ),
            ]
        ),
    ]

In [ ]:
app_package = ApplicationPackage(
    name="vespa",
    schema=[schema],
    components = components,
)

In [ ]:

# import os
# import docker

# # Set the DOCKER_HOST environment variable to point to the Docker Desktop socket
# # This is necessary because Docker Desktop on Linux uses a different socket path
# # from the default, and pyvespa/docker-py may not pick it up automatically.
# docker_desktop_socket_path = os.path.expanduser("~/.docker/desktop/docker.sock")
# if os.path.exists(docker_desktop_socket_path):
#     os.environ['DOCKER_HOST'] = f"unix://{docker_desktop_socket_path}"

# # Verify that we can connect to the Docker daemon
# try:
#     client = docker.from_env()
#     print("Successfully connected to Docker daemon.")
#     print(f"Docker version: {client.version()['Version']}")
# except Exception as e:
#     print(f"Failed to connect to Docker daemon: {e}")
#     print("\nPlease ensure Docker Desktop is running.")



In [ ]:
# Sempre que essa célula der erro execute o comando vespa auth login no terminal
# para autenticar novamente com o Vespa Cloud
vespa_cloud = VespaCloud(
    tenant=tenant_name,
    application=application,
    application_package=app_package,
)

# from vespa.deployment import VespaDocker
# from vespa.application import Vespa

# # vespa_docker = VespaDocker(container_memory= 24* 1024 ** 3,)
# app2 = Vespa(url="http://localhost", port=8080)
# app2.deploy()



Setting application...
Running: vespa config set application projetos-segundo.reranking.default
Setting target cloud...
Running: vespa config set target cloud

No api-key found for control plane access. Using access token.
Checking for access token in auth.json...
Successfully obtained access token for control plane access.
Certificate and key not found in /home/vinicius/Documents/GitHub/MSMARCO/vespa-search/.vespa or /home/vinicius/.vespa/projetos-segundo.reranking.default: Creating new cert/key pair with vespa CLI.
Generating certificate and key...
Running: vespa auth cert -N
Success: Certificate written to '/home/vinicius/.vespa/projetos-segundo.reranking.default/data-plane-public-cert.pem'
Success: Private key written to '/home/vinicius/.vespa/projetos-segundo.reranking.default/data-plane-private-key.pem'

Deployment started in run 1 of dev-aws-us-east-1c for projetos-segundo.reranking. This may take a few minutes the first time.
INFO    [15:18:25]  Deploying platform version 8.538

In [16]:
# Splita as queries de treinamento e teste

random.seed(42)

def load_dataset(input_file):
    with open(input_file, 'rb') as f:
        return pickle.load(f)

data_set = "../subset_msmarco_train_0.01_99.pkl"

data = load_dataset(data_set)
queries = data["queries"]
documents = data["docs"]
qrels = data["qrels"]

# Split the queries (queries is a dictionary of {query_id: query_object})
query_ids = list(queries.keys())  # List of query IDs

# Shuffle query IDs to ensure a random split
random.shuffle(query_ids)

# Split into 80% for training, 20% for validation
split_ratio = 0.8
train_query_ids = query_ids[:int(len(query_ids) * split_ratio)]
test_query_ids = query_ids[int(len(query_ids) * split_ratio):]

train_queries = {qid: queries[qid] for qid in train_query_ids}
test_queries = {qid: queries[qid] for qid in test_query_ids}

In [17]:
# remove control characters from the text to avoid issues with Vespa Feed

def remove_control_characters(text: str) -> str:
    """Remove caracteres de controle e não imprimíveis do texto."""
    return ''.join(
        ch for ch in text
        if unicodedata.category(ch)[0] != 'C' or ch in '\n\t\r'
    )

In [18]:
# Saves the data into a json format that Vespa can understand possivelmente we can remove this saving step 
# and create the json on memory just before the feed step

namespace = "default"
doctype = "doc"

vespa_docs = []

for doc_id, doc_obj in documents.items():
    vespa_doc = {
        "put": f"id:{namespace}:{doctype}::{doc_id}",
        "fields": {
            "id": str(doc_id),
            "text": remove_control_characters(doc_obj.text),
        }
    }
    vespa_docs.append(vespa_doc)

feed_file = "vespa_feed.json"

with open(feed_file, "w", encoding="utf-8") as f:
    json.dump(vespa_docs, f, ensure_ascii=False)

print(f"✅ {len(vespa_docs)} documentos salvos em {feed_file}")

✅ 277168 documentos salvos em vespa_feed.json


In [19]:
# Define feed parameters for the Vespa application
@dataclass
class FeedParams:
    name: str
    num_docs: int
    max_connections: int
    function_name: str
    max_workers: Optional[int] = None
    max_queue_size: Optional[int] = None


@dataclass
class FeedResult(FeedParams):
    feed_time: Optional[float] = None

In [20]:
# This is necessary to avoid issues with asyncio and Jupyter notebooks
nest_asyncio.apply()

In [21]:
# Asynchronous feed function that sends documents to Vespa
params = FeedParams(
    name="full_async_feed",
    function_name="feed_async_iterable",
    num_docs=0,  # não é usado aqui
    max_connections=16,
    max_workers=32,
    max_queue_size=5000,
)

# Load the documents from the JSON file
with open("vespa_feed.json", "r", encoding="utf-8") as f:
    data_list = json.load(f)

# Prepare the dataset for feeding into Vespa
dataset = [
    {"id": item["fields"]["id"], "fields": item["fields"]}
    for item in tqdm(data_list, desc="🔄 Preparando documentos para envio")
]

# Feed the dataset into Vespa asynchronously
with tqdm(total=len(dataset), desc="📤 Enviando documentos para Vespa") as pbar:
    def progress_callback(response, doc_id):
        pbar.update(1)
        if not response.is_successful():
            print(f"❌ Erro ao enviar {doc_id}: {response.status_code}")
            try:
                print("Response JSON:", response.get_json())
            except Exception as e:
                print("Could not get JSON from response:", e)
            try:
                print("Response text:", getattr(response, 'text', None))
            except Exception as e:
                print("Could not get text from response:", e)
            try:
                print("Exception:", getattr(response, 'exception', None))
            except Exception as e:
                print("Could not get exception from response:", e)

    start = time()
    app.feed_async_iterable(
        dataset,
        schema="doc",
        namespace="default",  # <-- use 'default' for consistency
        operation_type="feed",
        max_queue_size=params.max_queue_size,
        max_workers=params.max_workers,
        max_connections=params.max_connections,
        callback=progress_callback,
    )
    duration = time() - start

print(f"✅ Feed finalizado em {duration:.2f} segundos")


🔄 Preparando documentos para envio:   0%|          | 0/277168 [00:00<?, ?it/s]

📤 Enviando documentos para Vespa:   0%|          | 0/277168 [00:00<?, ?it/s]

❌ Erro ao enviar msmarco_passage_00_513463638: 599
Response JSON: {'Exception': '', 'id': 'msmarco_passage_00_513463638', 'message': 'Exception during feed_data_point'}
Response text: None
Exception: None
❌ Erro ao enviar msmarco_passage_00_513570905: 599
Response JSON: {'Exception': '', 'id': 'msmarco_passage_00_513570905', 'message': 'Exception during feed_data_point'}
Response text: None
Exception: None
❌ Erro ao enviar msmarco_passage_00_513580657: 599
Response JSON: {'Exception': '', 'id': 'msmarco_passage_00_513580657', 'message': 'Exception during feed_data_point'}
Response text: None
Exception: None
❌ Erro ao enviar msmarco_passage_00_513586995: 599
Response JSON: {'Exception': '', 'id': 'msmarco_passage_00_513586995', 'message': 'Exception during feed_data_point'}
Response text: None
Exception: None
❌ Erro ao enviar msmarco_passage_00_513612592: 599
Response JSON: {'Exception': '', 'id': 'msmarco_passage_00_513612592', 'message': 'Exception during feed_data_point'}
Response te

KeyboardInterrupt: 

In [ ]:
import requests

# Update the health check URL to use the deployed cloud instance

def check_vespa_health():
    try:
        # Use the app's endpoint instead of localhost
        health_url = f"{app.url}/state/v1/health"
        response = requests.get(health_url)
        if response.status_code == 200:
            print("Vespa health status:", response.json())
        else:
            print(f"Failed to get health status. Status code: {response.status_code}")
    except Exception as e:
        print("Error connecting to Vespa:", e)

check_vespa_health()

Vespa health status: {'time': 1750689245032, 'status': {'code': 'up'}, 'metrics': {'snapshot': {'from': 1750689184.878, 'to': 1750689244.883}, 'values': [{'name': 'requestsPerSecond', 'values': {'count': 13422, 'rate': 223.68135988667612}}, {'name': 'latencySeconds', 'values': {'average': 0.26285978244672925, 'sum': 3528.104, 'count': 13422, 'last': 0.058, 'max': 0.723, 'min': 0.037, 'rate': 223.68135988667612}}]}}


In [ ]:
test_queries_dict = {
    q.query_id: q.text
    for q in test_queries.values()
}

relevant_docs = dict()
for qrel in qrels:
    relevant_docs[qrel.query_id] = relevant_docs.get(qrel.query_id, set())
    relevant_docs[qrel.query_id].add(qrel.doc_id)

# Vespa query function for cross-encoder reranking

def create_crossencoder_query_fn(top_k=10, timeout=60):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        escaped_query_text = query_text.replace("'", "\'")
        return {
            "yql": f"select * from sources * where userQuery() limit {k}",
            "query": query_text,
            "ranking": "reranking",
            "ranking.features.query(q)": f"embed(e5, '{escaped_query_text}')",  # Use e5 for embeddings
            "ranking.features.query(q_tokens)": f"embed(tokenizer, '{escaped_query_text}')",  # Use tokenizer for tokens
            "timeout": f"{timeout}s",
            "ranking.softtimeout.enable": "false",
        }
    return query_fn

vespa_query_fn = create_crossencoder_query_fn(top_k=10)

# Evaluate using VespaEvaluator with the reranking profile
evaluator = VespaEvaluator(
    queries=test_queries_dict,
    relevant_docs=relevant_docs,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-crossencoder-reranking",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True
)

results = evaluator.run()

print("Results for cross-encoder reranking:")
print("Primary Metric:", evaluator.primary_metric)
print("Results:", results)

RetryError: RetryError[<Future at 0x7ef2e1b473e0 state=finished raised ReadTimeout>]

In [ ]:
# Test with just a few queries first
test_queries_sample = dict(list(test_queries_dict.items())[:5])  # Only 5 queries
relevant_docs_sample = {k: v for k, v in relevant_docs.items() if k in test_queries_sample}

evaluator = VespaEvaluator(
    queries=test_queries_sample,
    relevant_docs=relevant_docs_sample,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-crossencoder-reranking-sample",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True
)

results = evaluator.run()

RetryError: RetryError[<Future at 0x7ef2df631c40 state=finished raised ReadTimeout>]

In [ ]:
# Test a single query manually first to diagnose the issue
test_query = list(test_queries_dict.values())[0]
# print(f"Testing with query: {test_query}")

# # Create a simple query without reranking first
# simple_query = {
#     "yql": "select * from sources * where userQuery() limit 10",
#     "query": test_query,
#     "ranking": "bm25",  # Use simple BM25 ranking first
#     "timeout": "30s"
# }

# try:
#     print("Testing simple BM25 query...")
#     response = app.query(simple_query)
#     print(f"✅ Simple query successful! Got {response.number_documents_retrieved} hits")
#     if response.hits:
#         print(f"First hit relevance: {response.hits[0]['relevance']}")
# except Exception as e:
#     print(f"❌ Simple query failed: {e}")

In [ ]:
# Now test with reranking but much longer timeout
def create_crossencoder_query_fn_debug(top_k=10, timeout=300):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        escaped_query_text = query_text.replace("'", "\'")
        return {
            "yql": f"select * from sources * where userQuery() limit {k}",
            "query": query_text,
            "ranking": "reranking",
            "ranking.features.query(q)": f"embed(e5, '{escaped_query_text}')",  # Use e5 for embeddings
            "ranking.features.query(q_tokens)": f"embed(tokenizer, '{escaped_query_text}')",  # Use tokenizer for tokens
            "timeout": f"{timeout}s",
            "ranking.softtimeout.enable": "false",
            "trace.level": "1",
            "presentation.timing": "true"
        }
    return query_fn

# Test reranking query with very long timeout
rerank_query_fn = create_crossencoder_query_fn_debug(top_k=5, timeout=300)  # Reduce to top 5 and 5 min timeout
rerank_query = rerank_query_fn(test_query)

try:
    print("Testing reranking query with extended timeout...")
    response = app.query(rerank_query)
    print(f"✅ Reranking query successful! Got {response.number_documents_retrieved} hits")
    if response.hits:
        print(f"First hit relevance: {response.hits[0]['relevance']}")
    
    # Print timing information if available
    if hasattr(response, 'json') and 'timing' in response.json:
        print(f"Query timing: {response.json['timing']}")
        
except Exception as e:
    print(f"❌ Reranking query failed: {e}")

Testing reranking query with extended timeout...
✅ Reranking query successful! Got 989 hits
First hit relevance: 0.03730776474501788
Query timing: {'querytime': 32.338, 'summaryfetchtime': 0.964, 'searchtime': 33.303}


In [ ]:
# Simple manual evaluation approach that doesn't rely on vespa.evaluation imports
def simple_evaluation(app, queries_dict, relevant_docs, query_fn, k=10):
    """
    Simple evaluation that calculates basic metrics manually
    """
    total_queries = 0
    total_relevant_retrieved = 0
    total_relevant = 0
    total_retrieved = 0
    reciprocal_ranks = []
    ndcg_scores = []
    
    print(f"Running simple evaluation on {len(queries_dict)} queries...")
    
    for query_id, query_text in tqdm(queries_dict.items(), desc="Processing queries"):
        try:
            # Execute query
            query_body = query_fn(query_text)
            response = app.query(query_body, timeout=120)
            
            if response.number_documents_retrieved > 0:
                # Get relevant documents for this query
                query_relevant_docs = relevant_docs.get(query_id, set())
                
                if query_relevant_docs:
                    total_queries += 1
                    total_relevant += len(query_relevant_docs)
                    
                    # Get retrieved document IDs
                    retrieved_docs = [hit['fields']['id'] for hit in response.hits[:k]]
                    total_retrieved += len(retrieved_docs)
                    
                    # Count relevant retrieved
                    relevant_retrieved = len(set(retrieved_docs) & query_relevant_docs)
                    total_relevant_retrieved += relevant_retrieved
                    
                    # Calculate reciprocal rank
                    rr = 0
                    for i, doc_id in enumerate(retrieved_docs):
                        if doc_id in query_relevant_docs:
                            rr = 1.0 / (i + 1)
                            break
                    reciprocal_ranks.append(rr)
                    
                    # Calculate NDCG@k
                    dcg = 0
                    idcg = sum([1.0 / (i + 1) for i in range(min(len(query_relevant_docs), k))])
                    for i, doc_id in enumerate(retrieved_docs):
                        if doc_id in query_relevant_docs:
                            dcg += 1.0 / (i + 1)
                    ndcg = dcg / idcg if idcg > 0 else 0
                    ndcg_scores.append(ndcg)
                    
        except Exception as e:
            print(f"Error processing query {query_id}: {e}")
            continue
    
    # Calculate metrics
    precision = total_relevant_retrieved / total_retrieved if total_retrieved > 0 else 0
    recall = total_relevant_retrieved / total_relevant if total_relevant > 0 else 0
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0
    ndcg = sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0
    
    return {
        f'precision_at_{k}': precision,
        f'recall_at_{k}': recall,
        'mrr': mrr,
        f'ndcg_at_{k}': ndcg,
        'total_queries': total_queries,
        'avg_relevant_retrieved': total_relevant_retrieved / total_queries if total_queries > 0 else 0
    }

# Test simple evaluation with cross-encoder
print("Testing simple manual evaluation with cross-encoder...")
test_queries_tiny = dict(list(test_queries_dict.items())[:2])
relevant_docs_tiny = {k: v for k, v in relevant_docs.items() if k in test_queries_tiny}

conservative_query_fn = create_crossencoder_query_fn_debug(top_k=5, timeout=300)

try:
    simple_results = simple_evaluation(
        app=app,
        queries_dict=test_queries_tiny,
        relevant_docs=relevant_docs_tiny,
        query_fn=conservative_query_fn,
        k=5
    )
    print("✅ Simple cross-encoder evaluation successful!")
    print("Cross-encoder Results:", simple_results)
except Exception as e:
    print(f"❌ Simple cross-encoder evaluation failed: {e}")

Testing simple manual evaluation with cross-encoder...
Running simple evaluation on 2 queries...


Processing queries:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Simple cross-encoder evaluation successful!
Cross-encoder Results: {'precision_at_5': 0.1, 'recall_at_5': 0.5, 'mrr': 0.5, 'ndcg_at_5': 0.5, 'total_queries': 2, 'avg_relevant_retrieved': 0.5}
